In [ ]:
%pip install kafi==1.1.5
import sys
sys.path.insert(1, "../..")

In [ ]:
# python3 data-gen.py --start-date "2026-10-30 00:00:00" --end-date "2026-10-31 00:00:00" --topic metrics-raw-telegraf-env --bootstrap-server localhost:9092

In [1]:
from kafi.streams.streams import Streams
import logging
logging.basicConfig(level=logging.INFO)

/home/ralph/github/kcc-kafi/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}, "kafi": {"consume.timeout": 0.1}})

In [ ]:
# Sanity check: see if we can connect to Kafka
c.l()

In [11]:
# Setup: Produce a context event to the context-data topic
TOPIC_NAMES = [
    "accounting",
    # "analytics",
    # "api",
    # "auth",
    # "billing",
    # "catalog",
    # "checkout",
    # "customer",
    # "inventory",
    # "loyalty",
    # "orders",
    # "payments",
    # "products",
    # "recommendations",
    # "shipping",
    # "subscriptions",
]
context_topic = "context-data"

c.retouch(context_topic)

p = c.producer(context_topic)

for t in TOPIC_NAMES:
    event = {
        "id": "323b603d-5b5f-48d2-84fc-4e784e942289",
        "entityType": "TOPIC",
        "regex": f".*{t}.*",
        "context": [
            {
                "key": "application",
                "value": t
            },
            {
                "key": "psp_element",
                "value": "B109968_LAGO D&A | Team $1 - $1@mgb.ch |"
            }
        ]
    }
    p.produce(event, key="k")

p.close()

'context-data'

## Specify the Topology

In [22]:
import copy, re

from kafi.streams.streams import Streams
import logging
logging.basicConfig(level=logging.INFO)

from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}, "kafi": {"consume.timeout": 0.1, "consume.batch.size": 100}})

context_source = "context-data"
metric_source = "metrics-raw-telegraf-env"

output_topic = "test-topic"

c.retouch(output_topic)

# {
#     "id": "323b603d-5b5f-48d2-84fc-4e784e942289",
#     "entityType": "TOPIC",
#     "regex": "(.+)",
#     "context": [
#        {
#            "key": "application",
#            "value": "$1"
#        },
#        {
#            "key": "psp_element",
#            "value": "B109968_LAGO D&A | Team $1 - $1@mgb.ch |"
#        }
#    ]
#}
contexts = (
    Streams.source(c, context_source)
    .map(lambda r: r["value"])
)

# {
#  "fields": {
#    "counter_diff": 12198
#  },
#  "name": "kafka_server_brokertopicmetrics_bytesout_total",
#  "tags": {
#    "env": "dev",
#    "pod_name": "poc-cluster-dual-role-2",
#    "topic": "kcc-test-aggregated"
#  },
#  "timestamp": 1746799800
# }
def ts_fun(r):
    return r["timestamp"] * 1000
size_int = 1000 * 60 * 60

metrics = (
    Streams.source(c, metric_source)
    .map(lambda r: r["value"])
)

metrics_expired = (
    metrics.
    expire_tumbling(ts_fun=ts_fun, size_int=size_int, allowed_lateness_int=1000 * 1000 * 60 * 1) # 1 hour, allowed_lateness_int=1 minute
)

# joined = (
#     contexts
#    .join(metrics_expired,
#          left_key_fun=lambda c: c["regex"],
#          right_key_fun=lambda m: m["tags"]["topic"],
#          project_fun=lambda c, m: {"context": {"entityType": c["entityType"],
#                                             "regex": c["regex"],
#                                             "context": c["context"]},
#                                     "metric": {"fields": m["fields"],
#                                             "name": m["name"],
#                                             "tags": {"topic": m["tags"]["topic"]}},
#                                     "timestamp": m["timestamp"]})
# )
def predicate(regex, topic):
    pattern = re.compile(regex)
    return pattern.match(topic) is not None

joined = (
    contexts
   .join_pred(metrics_expired,
        #  left_key_fun=lambda c: c["regex"],
        #  right_key_fun=lambda m: m["tags"]["topic"],
         predicate_fun=lambda c, m: predicate(c["regex"], m["tags"]["topic"]),
         project_fun=lambda c, m: {"context": {"entityType": c["entityType"],
                                            "regex": c["regex"],
                                            "context": c["context"]},
                                    "metric": {"fields": m["fields"],
                                            "name": m["name"],
                                            "tags": {"topic": m["tags"]["topic"]}},
                                    "timestamp": m["timestamp"]})
)

def get_app_context(joined_r):
    ctx = joined_r["context"]
    app = "unknown_app"
    metric_name = joined_r["metric"]["name"]
    for c in ctx["context"]:
        if c["key"] == "application":
            app = c["value"]
    return app + "|" + metric_name

def combine_metrics(agg_r, r):
    """
    Generates a deep-copy of agg_r and adds the counter_diff of r to the counter_diff of agg_r.
    :param agg_r:
    :param r:
    :return:
    """
    agg_r = copy.deepcopy(agg_r)
    agg_r["fields"]["counter_diff"] += r["fields"]["counter_diff"]
    agg_r["fields"]["counter_diffs"] += [r["fields"]["counter_diff"]]
    return agg_r

def agg_fun(agg_r, r):
    x = {"context": r["context"]["context"],
         "metric": combine_metrics(agg_r["metric"], r["metric"]),
         "timestamps": agg_r.get("timestamps", []) + [r["timestamp"]]}
    return x

def proj_fun(key, agg):
    x = {"agg": agg, "key": key}
    return x

windowed = joined.group_by_agg_tumbling(ts_fun=ts_fun,
                             key_fun=get_app_context,
                             size_int=size_int,
                             agg_fun=agg_fun,
                             agg_initial_any={"context": dict(),
                                              "metric": {"fields": {"counter_diff": 0, "counter_diffs": []}},
                                              "timestamps": []},
                             project_fun=proj_fun
                             )
                             
sink = windowed.map(lambda r: {"value": r}).sink(c, output_topic)

topology = Streams.build(sink)

#stop = Streams.start_streams(topology, progress=True)
Streams.streams(topology, progress=True)


(['metrics-raw-telegraf-env'], 'streams_1793977389625')
(['context-data'], 'streams_1793977389625')
Uptime: 0.213s, State size: 304.25 KB, Source offsets: {'metrics-raw-telegraf-env': {0: 1280}, 'context-data': {0: 1}}, Sink outputs: 8

KeyboardInterrupt: 

In [ ]:
print(Streams.threads())
stop()
Streams.threads()

In [ ]:
c.l()